##### Part 2 — Set up Unity Catalog objects
Referring to code snippets from Python program: 
[notebooks/00_setup_unity_catalog.py].

[1] - Create UNITY CATALOG volume: The path

In [0]:
# SETTING CONSTANTS ACCORDING TO:
# workspace.prj_fintech-transaction-reporting-on-databricks-with-spark_sc/
catalog_name = "workspace"
schema_name = "prj_fintech-transaction-reporting-on-databricks-with-spark_sc"
volume_name = "raw_data"

raw_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

print("Raw volume path:", raw_volume_path)

[2] - Create UNITY CATALOG volume: The 3-layered objects 
1. Catalog: workspace, 
2. Schema: prj_fintech-transaction-reporting-on-databricks-with-spark_sc
3. Volume: raw_data

NOTE: since the SCHEMA has a complicated formatting with "-" and "_", it must be "back quoted" with "`" characters!

In [0]:
# CREATING THE UNITY CATALOG volume related structure elements:
# 1. Run SQL to create the catalog if needed.
# 2. Run SQL to create the schema if needed.
# 3. Run SQL to create the volume if needed.

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.`{schema_name}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.`{schema_name}`.{volume_name}")

[3] - Create SETTINGS from Project REPO:
1. path to the delivered source files
2. list of the delivered source files

In [0]:
# PATH: SETTING CONSTANT ACCORDING TO:
# /Workspace/Users/michael.c.feiertag@web.de/PRJ_fintech-transaction-reporting-on-databricks-with-spark/sample_data/
repo_sample_path = "file:/Workspace/Users/michael.c.feiertag@web.de/PRJ_fintech-transaction-reporting-on-databricks-with-spark/sample_data"
print("Repo File`s path:", repo_sample_path)

# FILES: create Python List with source files:
files_to_copy = ["transactions_sample.csv", "customers_sample.csv", "accounts_sample.csv"]
print("Repo Files to copy:", files_to_copy)


[4] - Execute COPY OF FILES from Project REPO into UNITY CATALOG volume

Note: cut-off the "sample" name from the original filename when storing into UNITY CATALOG volume.


In [0]:
dbutils.fs.mkdirs(raw_volume_path)

# Copy the three sample CSV files from the repo folder into the UC volume.
for file_name in files_to_copy:
    source = f"{repo_sample_path}/{file_name}"
    target_name = file_name.replace("_sample", "")
    target = f"{raw_volume_path}/{target_name}"
    print(f"Copying {source} -> {target}")
    try:
        dbutils.fs.cp(source, target)
    except Exception as e:
        print("Copy skipped or failed:", e)

# Verify successful copy
display(dbutils.fs.ls(raw_volume_path)) 